# ANCP T## — D3 Exploratory Data Analysis (EDA)
## IndiGo EBITDA Margin and macro-market indicators **Dataset used:** `EDA ANALYSIS(5).xlsx` — `Sheet1` **Observation level:** 38 quarterly observations **Primary and only profitability outcome:** EBITDA Margin This notebook follows the same broad EDA workflow as the supplied EDA notebook, adapted to the new quarterly dataset. The analysis focuses exclusively on EBITDA Margin and its relationship with USD/INR, CPI, WPI, and the supplied oil benchmark indicators. **Business question:** How are IndiGo's quarterly EBITDA margins associated with the supplied macroeconomic and oil-market indicators, and what does the evidence imply for next-quarter EBITDA-margin modelling? > Associations are exploratory and do not establish causation. The workbook does not identify the exact oil benchmark or units, so the oil variables are treated as supplied benchmark proxies.

## 1. Business question and analytical objectives The analysis will:
1. assess the structure, completeness, consistency, and coverage of the quarterly dataset;
3. examine profitability trends over time;
4. explore relationships between profitability and macro/market indicators;
5. compare calendar-quarter patterns;
6. test pre-defined exploratory hypotheses;
7. record evidence-based insights and implications for the next-quarter EBITDA-margin model. The dataset does not include passenger load factor, passenger yield, capacity, revenue, fuel consumption, or explicit source metadata. These omissions limit the business explanations that can be supported.

## 2. Hypotheses defined before viewing analytical results - **H2:** EBITDA Margin is associated with USD/INR.
- **H3:** EBITDA Margin is associated with the oil benchmark.
- **H4:** EBITDA Margin is negatively associated with CPI inflation.
- **H5:** EBITDA Margin is negatively associated with WPI inflation.
- **H6:** EBITDA Margin differs across calendar quarters. These are exploratory hypotheses. Statistical significance should be interpreted cautiously because the dataset contains only 38 quarterly observations and time-series observations may not be independent.

## 3. Imports, configuration, and reproducibility

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA_FILE = Path("EDA ANALYSIS(5).xlsx")
if not DATA_FILE.exists():
    candidates = [Path("/content/EDA ANALYSIS(5).xlsx"),
                  Path("/mnt/data/EDA ANALYSIS(5).xlsx")]
    DATA_FILE = next((p for p in candidates if p.exists()), None)

if DATA_FILE is None:
    raise FileNotFoundError(
        "Upload 'EDA ANALYSIS(5).xlsx' to the notebook working directory "
        "or update DATA_FILE with its correct path."
    )

print("Dataset path:", DATA_FILE.resolve())

## 4. Dataset loading and initial inspection

In [ ]:
raw = pd.read_excel(DATA_FILE, sheet_name="Sheet1")
df = raw.copy()

print("Dataset shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nData types:")
display(df.dtypes.rename("data_type").to_frame())
print("\\nFirst five rows:")
display(df.head())
print("\\nLast five rows:")
display(df.tail())
print("\\nDescriptive overview:")
display(df.describe(include="all").T)

## 5. Original-data quality assessment

In [ ]:
def audit_frame(frame, name):
    return {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
        "missing_pct": round(frame.isna().sum().sum() / max(frame.size, 1) * 100, 2),
    }

display(pd.DataFrame([audit_frame(raw, "EDA ANALYSIS(5).xlsx — Sheet1")]))
display(pd.DataFrame({
    "column": raw.columns,
    "dtype": [str(raw[c].dtype) for c in raw.columns],
    "missing_count": [int(raw[c].isna().sum()) for c in raw.columns],
    "unique_values": [int(raw[c].nunique(dropna=True)) for c in raw.columns],
}))

## 6. Original-data representativeness and bias checks

In [ ]:
print("Year coverage:")
display(df.groupby("Year").size().rename("quarterly_rows").to_frame())

print("Calendar-quarter coverage:")
display(df["Quarter (Calendar)"].value_counts(dropna=False).sort_index().rename("rows").to_frame())

print("Year and quarter combinations:")
display(df[["Year", "Quarter (Calendar)"]].head(12))

print("Potential duplicate year-quarter combinations:",
      int(df.duplicated(["Year", "Quarter (Calendar)"]).sum()))

## 7. Documented cleaning and transformation

In [ ]:
cleaning_log = []

def log_cleaning(action, reason, impact):
    cleaning_log.append({
        "dataset": DATA_FILE.name,
        "action": action,
        "reason": reason,
        "impact": impact
    })

panel = raw.copy()

# Standardize column names only for easier, consistent analysis.
column_map = {
    "Quarter (Calendar)": "calendar_quarter",
    : "EBITDA Margin": "ebitda_margin_pct",
    "USD INR (Quarterly Mean)": "usd_inr",
    "CPI inflation (Quarterly Mean)": "cpi_inflation",
    "WPI inflation (Quarterly Mean)": "wpi_inflation",
    "Oil benchmark(DAILY) (Quarterly Mean)": "oil_daily_mean",
    "Oil benchmark(MONTHLY) (Quarterly Mean)": "oil_monthly_mean",
}
panel = panel.rename(columns=column_map)
log_cleaning("Standardized column names",
             "Create consistent analysis-friendly variable names",
             "Column labels changed; underlying values retained")

numeric_cols = [
    "Year", "ebitda_margin_pct", "usd_inr",
    "cpi_inflation", "wpi_inflation", "oil_daily_mean", "oil_monthly_mean"
]
for col in numeric_cols:
    before_missing = int(panel[col].isna().sum())
    panel[col] = pd.to_numeric(panel[col], errors="coerce")
    after_missing = int(panel[col].isna().sum())
    if after_missing > before_missing:
        log_cleaning(f"Numeric conversion: {col}",
                     "Convert values to numeric for statistical analysis",
                     f"{after_missing-before_missing} additional missing value(s) created")

quarter_order = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
panel["quarter_num"] = panel["calendar_quarter"].astype(str).str.upper().map(quarter_order)
panel["quarter_end"] = pd.to_datetime(
    panel["Year"].astype("Int64").astype(str) + "-" +
    panel["quarter_num"].map({1:"03-31", 2:"06-30", 3:"09-30", 4:"12-31"}),
    errors="coerce"
)
log_cleaning("Created quarter-end date",
             "Convert Year and calendar quarter into a sortable date",
             "Added quarter_end and quarter_num fields")

panel = panel.sort_values("quarter_end").reset_index(drop=True)
log_cleaning("Chronological sorting",
             "Make time-based patterns and lag order readable",
             "Rows reordered chronologically; observations retained")

# No automatic outlier removal, winsorization, or missing-value imputation.
log_cleaning("Outlier handling",
             "Retain extreme observations for contextual review",
             "No values removed or capped")
log_cleaning("Missing-value handling",
             "Avoid introducing unsupported assumptions",
             "No imputation performed")

display(pd.DataFrame(cleaning_log))

## 8. Cleaned-data quality checks

In [ ]:
display(pd.DataFrame([audit_frame(panel, "Quarterly analysis panel")]))
display(pd.DataFrame({
    "variable": panel.columns,
    "dtype": [str(panel[c].dtype) for c in panel.columns],
    "missing_count": [int(panel[c].isna().sum()) for c in panel.columns],
    "unique_values": [int(panel[c].nunique(dropna=True)) for c in panel.columns],
}))
print("Chronological order check:",
      bool(panel["quarter_end"].is_monotonic_increasing))
print("Duplicate quarter-end dates:", int(panel["quarter_end"].duplicated().sum()))

## 9. Repeated representativeness and bias checks on cleaned data

In [ ]:
coverage = pd.DataFrame([{
    "dataset": "Quarterly analysis panel",
    "rows": len(panel),
    "start_quarter": panel["quarter_end"].min(),
    "end_quarter": panel["quarter_end"].max(),
    "calendar_quarters_present": panel["calendar_quarter"].nunique(),
    "missing_EBITDA": int(panel["ebitda_margin_pct"].isna().sum()),
    "missing_PAT": int(panel[].isna().sum()),
}])
display(coverage)

print("Rows per calendar quarter:")
display(panel.groupby("calendar_quarter").size().rename("observations").to_frame())

print("Representativeness notes:")
print("- The dataset contains quarterly observations only.")
print("- The available period and indicators may not represent all drivers of airline profitability.")
print("- There are no operational variables such as load factor, passenger yield, or capacity.")
print("- The oil benchmark definition and units are not specified in the workbook.")

## 10. Distribution analysis

In [ ]:
num_cols = [
    "ebitda_margin_pct", "usd_inr",
    "cpi_inflation", "wpi_inflation", "oil_daily_mean", "oil_monthly_mean"
]
display(panel[num_cols].describe().T)

distribution_stats = []
for col in num_cols:
    values = panel[col].dropna()
    distribution_stats.append({
        "variable": col,
        "count": len(values),
        "skewness": values.skew() if len(values) > 2 else np.nan,
        "kurtosis": values.kurtosis() if len(values) > 3 else np.nan,
        "IQR": values.quantile(.75) - values.quantile(.25) if len(values) else np.nan
    })
display(pd.DataFrame(distribution_stats))

for col in ["ebitda_margin_pct"]:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(panel[col].dropna(), bins=10, edgecolor="black", alpha=0.75)
    ax.set_title(f"Distribution of {col.replace('_', ' ').title()}")
    ax.set_xlabel("Margin (%)")
    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=panel[num_cols], orient="h", ax=ax)
ax.set_title("Outlier screening across numeric variables")
ax.set_xlabel("Value (variables have different scales)")
plt.tight_layout()
plt.show()

## 11. Time trends and relationship analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(panel["quarter_end"], panel["ebitda_margin_pct"], marker="o", label="EBITDA margin (%)")
ax.set_title("Quarterly profitability margins over time")
ax.set_xlabel("Quarter")
ax.set_ylabel("Margin (%)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

macro_cols = ["usd_inr", "cpi_inflation", "wpi_inflation", "oil_daily_mean", "oil_monthly_mean"]
fig, axes = plt.subplots(len(macro_cols), 1, figsize=(12, 14), sharex=True)
for ax, col in zip(axes, macro_cols):
    ax.plot(panel["quarter_end"], panel[col], marker="o")
    ax.set_ylabel(col.replace("_", " "))
    ax.grid(True, alpha=.3)
axes[-1].set_xlabel("Quarter")
fig.suptitle("Macroeconomic and oil benchmark indicators over time", y=1.01)
plt.tight_layout()
plt.show()

corr_cols = ["ebitda_margin_pct"] + macro_cols
corr = panel[corr_cols].corr(method="spearman")
display(corr.round(3))
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax)
ax.set_title("Spearman correlation matrix")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, col in zip(axes.flat, macro_cols):
    sns.regplot(data=panel, x=col, y="ebitda_margin_pct",
                lowess=True, scatter_kws={"alpha":.75}, line_kws={"color":"red"}, ax=ax)
    ax.set_title(f"EBITDA margin vs {col.replace('_', ' ')}")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

## 12. Segment analysis

In [ ]:
quarter_summary = (
    panel.groupby("calendar_quarter")["ebitda_margin_pct"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reindex(["Q1", "Q2", "Q3", "Q4"])
)
display(quarter_summary)

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=panel, x="calendar_quarter", y="ebitda_margin_pct",
            order=["Q1", "Q2", "Q3", "Q4"], ax=ax)
sns.stripplot(data=panel, x="calendar_quarter", y="ebitda_margin_pct",
              order=["Q1", "Q2", "Q3", "Q4"], color="black", alpha=.55, ax=ax)
ax.set_title("EBITDA margin by calendar quarter")
ax.set_xlabel("Calendar quarter")
ax.set_ylabel("EBITDA margin (%)")
plt.tight_layout()
plt.show()

pat_quarter_summary = (
    panel.groupby("calendar_quarter")[]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reindex(["Q1", "Q2", "Q3", "Q4"])
)
display(pat_quarter_summary)

## 13. Hypothesis assessment and statistical tests

In [ ]:
hypothesis_results = []

def spearman_test(hid, statement, x, y, expected_direction=None):
    sub = panel[[x, y]].dropna()
    if len(sub) < 5 or sub[x].nunique() < 2 or sub[y].nunique() < 2:
        hypothesis_results.append({
            "ID": hid, "Hypothesis": statement, "n": len(sub),
            "Spearman_rho": np.nan, "p_value": np.nan,
            "Result": "Insufficient usable variation/data"
        })
        return
    rho, p = stats.spearmanr(sub[x], sub[y])
    if p < .05:
        result = "Statistically significant exploratory association"
    else:
        result = "No statistically significant association detected"
    if expected_direction and np.sign(rho) != expected_direction:
        result += "; observed direction differs from expectation"
    hypothesis_results.append({
        "ID": hid, "Hypothesis": statement, "n": len(sub),
        "Spearman_rho": rho, "p_value": p, "Result": result
    })

              "ebitda_margin_pct", expected_direction=1)
spearman_test("H2", "EBITDA margin is associated with USD/INR",
              "ebitda_margin_pct", "usd_inr")
spearman_test("H3", "EBITDA margin is positively associated with oil benchmark",
              "ebitda_margin_pct", "oil_monthly_mean", expected_direction=1)
spearman_test("H4", "EBITDA margin is negatively associated with CPI inflation",
              "ebitda_margin_pct", "cpi_inflation", expected_direction=-1)
spearman_test("H5", "EBITDA margin is negatively associated with WPI inflation",
              "ebitda_margin_pct", "wpi_inflation", expected_direction=-1)

# Kruskal-Wallis test compares EBITDA margin distributions across calendar quarters.
groups = [
    panel.loc[panel["calendar_quarter"] == q, "ebitda_margin_pct"].dropna().values
    for q in ["Q1", "Q2", "Q3", "Q4"]
]
groups = [g for g in groups if len(g) > 0]
if len(groups) >= 2 and all(len(g) > 0 for g in groups):
    h_stat, p_value = stats.kruskal(*groups)
    hypothesis_results.append({
        "ID": "H6",
        "Hypothesis": "EBITDA margin differs across calendar quarters",
        "n": int(sum(len(g) for g in groups)),
        "Spearman_rho": np.nan,
        "p_value": p_value,
        "Result": "Quarter-group difference detected" if p_value < .05
                  else "No statistically significant quarter-group difference detected"
    })

hypothesis_table = pd.DataFrame(hypothesis_results)
display(hypothesis_table)
print("Caution: p-values are exploratory; small sample size and time dependence limit inference.")

## 14. Insight log

In [ ]:
insights = []

def add_insight(iid, insight, evidence, business, modelling, confidence):
    insights.append({
        "Insight ID": iid,
        "Insight": insight,
        "Evidence": evidence,
        "Business implication": business,
        "Modelling implication": modelling,
        "Confidence": confidence
    })

# Generate evidence descriptions directly from the dataset rather than hard-coding results.
ebitda_corr_pat = panel[["ebitda_margin_pct"]].corr(method="spearman").iloc[0,1]
add_insight(
    "I1",
    f"Spearman correlation = {ebitda_corr_pat:.3f} based on available paired quarters.",
    "Operating profitability and final profitability are related but are not identical measures.",
    "Medium; descriptive association only"
)

for col, label in [
    ("usd_inr", "USD/INR"),
    ("cpi_inflation", "CPI inflation"),
    ("wpi_inflation", "WPI inflation"),
    ("oil_monthly_mean", "monthly oil benchmark proxy")
]:
    sub = panel[["ebitda_margin_pct", col]].dropna()
    if len(sub) >= 5:
        rho, p = stats.spearmanr(sub["ebitda_margin_pct"], sub[col])
        add_insight(
            f"I{len(insights)+1}",
            f"EBITDA margin has an observed rank association with {label}.",
            f"Spearman rho = {rho:.3f}; p-value = {p:.3f}; n = {len(sub)}.",
            "The indicator may provide context for profitability changes, but the association is not causal.",
            "Retain as a candidate feature for validation; avoid assuming a stable relationship.",
            "Low to medium; small quarterly sample"
        )

display(pd.DataFrame(insights))

## 15. Implications for future modelling

In [ ]:
model_recommendations = pd.DataFrame([
    ["Baseline", "Compare every model with a simple historical and seasonal-naive benchmark.", "Prevents complex models from appearing useful without evidence."],
    ["Validation", "Use chronological or expanding-window validation; do not randomly shuffle quarterly observations.", "Preserves time order and reduces look-ahead risk."],
    ["Features", "Evaluate USD/INR, CPI, WPI, and both supplied oil benchmark series as candidate predictors.", "Use only variables available at the forecast date."],
    ["Small sample", "Prefer parsimonious models and report uncertainty.", "Only 38 quarterly observations are available."],
    ["Leakage control", "Any lagged or rolling features must use information available at the forecast origin.", "Prevents future information entering model training."],
    ["Further data", "Add operational drivers such as load factor, passenger yield, capacity, and fuel cost if reliable data becomes available.", "Current workbook omits important airline operating drivers."]
], columns=["Area", "Recommendation", "Reason"])
display(model_recommendations)

## 16. Limitations and uncertainty

- The dataset contains only 38 quarterly observations, limiting statistical power and machine-learning reliability.
- The workbook is already quarterly aggregated; the original daily/monthly source records are not available here for independent aggregation checks.
- The exact oil benchmark, source, and units are not specified. It should not automatically be interpreted as IndiGo's actual aviation turbine fuel cost.
- The dataset does not include passenger demand, load factor, passenger yield, capacity, revenue, or route mix.
- Correlation and statistical tests identify associations, not causal effects.
- Quarterly observations may be serially dependent, which can affect conventional statistical test assumptions.
- No missing-value imputation or outlier removal is performed; analyses use available observations.
- The analysis depends on the accuracy and definitions of the supplied workbook fields.

## 17. Conclusions and next steps

In [ ]:
print("FINAL EDA CHECKLIST")
print("Dataset:", DATA_FILE.name)
print("Rows:", len(panel))
print("Columns after transformation:", panel.shape[1])
print("Quarterly EBITDA observations:", int(panel["ebitda_margin_pct"].notna().sum()))
print("Quarter span:", panel["quarter_end"].min(), "to", panel["quarter_end"].max())
print("Duplicate year-quarter combinations:", int(panel.duplicated(["Year", "calendar_quarter"]).sum()))
print("Missing values by variable:")
display(panel.isna().sum().rename("missing_count").to_frame())

### Conclusion This EDA applies the same broad analytical workflow as the supplied notebook to the new `EDA ANALYSIS(5).xlsx` dataset. It examines the quality and coverage of 38 quarterly observations, describes profitability and macro-market variables, explores time trends and quarter-level patterns, and tests exploratory relationships.